In [3]:
from datetime import timedelta
from datetime import date
import comtradeapicall

# set some variables
# comtrade api subscription key (from comtradedeveloper.un.org), some preview and metadata/reference API calls do not require key
subscription_key = 'APICOMTRADEPREMINST'
directory = 'E:\Projects\vacoda\data\api'  # output directory for downloaded files
#proxy_url = '<PROXY URL>'  # optional if you need a proxy server

<>:8: SyntaxWarning: invalid escape sequence '\P'
<>:8: SyntaxWarning: invalid escape sequence '\P'
C:\Users\saxena\AppData\Local\Temp\ipykernel_21708\103920129.py:8: SyntaxWarning: invalid escape sequence '\P'
  directory = 'E:\Projects\vacoda\data\api'  # output directory for downloaded files


In [6]:
# set some variables again
today = date.today()
yesterday = today - timedelta(days=1)
lastweek = today - timedelta(days=7)

# C:commodities
# FreqCode: M (month), A : Anual
mydf = comtradeapicall.previewFinalData(typeCode='C', freqCode='A', clCode='HS', period='202205',
                                        reporterCode=None, cmdCode='441012', flowCode='M', partnerCode=None,
                                        partner2Code=None,
                                        customsCode=None, motCode=None, format_output='JSON',
                                        aggregateBy=None, breakdownMode='classic', countOnly=None, includeDesc=True)
mydf.head(5)

""


In [7]:
########### Python 3.2 #############
import urllib.request, json

try:
    url = "https://comtradeapi.un.org/bulk/v1/get/C/A/HS"

    hdr ={
    # Request headers
    'Cache-Control': 'no-cache',
    'Ocp-Apim-Subscription-Key': '••••••••••••••••••••••••••••••••',
    }

    req = urllib.request.Request(url, headers=hdr)

    req.get_method = lambda: 'GET'
    response = urllib.request.urlopen(req)
    print('suscessful', response.getcode())
    print(response.read())
except Exception as e:
    print(e)
####################################

'latin-1' codec can't encode characters in position 0-31: ordinal not in range(256)


In [11]:
from pathlib import Path
import pandas as pd

# Notebook runs from notebooks/; resolve project-root data path robustly.
candidates = [
    Path("../data/api/comtrade.parquet"),
    Path("data/api/comtrade.parquet"),
]

parquet_path = next((p for p in candidates if p.exists()), None)
if parquet_path is None:
    checked = "\n".join(str(p.resolve()) for p in candidates)
    raise FileNotFoundError(f"Parquet file not found. Checked:\n{checked}")

df_comtrade = pd.read_parquet(parquet_path)
print(f"Loaded: {parquet_path.resolve()}")
print(f"Shape: {df_comtrade.shape[0]:,} rows x {df_comtrade.shape[1]:,} columns")
df_comtrade.head()

Loaded: E:\Projects\vacoda\data\api\comtrade.parquet
Shape: 6,119,630 rows x 1 columns


,datasetCode\ttypeCode\tfreqCode\trefPeriodId\trefYear\trefMonth\tperiod\treporterCode\tflowCode\tpartnerCode\tpartner2Code\tclassificationSearchCode\tclassificationCode\tisOriginalClassification\tcmdCode\tcustomsCode\tmosCode\tmotCode\tqtyUnitCode\tqty\tisQtyEstimated\taltQtyUnitCode\taltQty\tisAltQtyEstimated\tnetWgt\tisNetWgtEstimated\tgrossWgt\tisGrossWgtEstimated\tCIFValue\tFOBValue\tprimaryValue\tlegacyEstimationFlag\tisReported\tisAggregate
0,20100201201092100\tC\tA\t20120101\t2012\t52\t2...
1,20100201201092100\tC\tA\t20120101\t2012\t52\t2...
2,20100201201092100\tC\tA\t20120101\t2012\t52\t2...
3,20100201201092100\tC\tA\t20120101\t2012\t52\t2...
4,20100201201092100\tC\tA\t20120101\t2012\t52\t2...


In [ ]:
# Convert 1-column tab-delimited payload into a properly structured table.
raw_col = df_comtrade.columns[0]
column_names = raw_col.split("\t")

# Split each row string by tab into separate columns.
df_comtrade_structured = df_comtrade[raw_col].str.split("\t", expand=True)

# Ensure final column count matches parsed header count.
if df_comtrade_structured.shape[1] != len(column_names):
    if df_comtrade_structured.shape[1] < len(column_names):
        missing = len(column_names) - df_comtrade_structured.shape[1]
        for i in range(missing):
            df_comtrade_structured[df_comtrade_structured.shape[1] + i] = None
    else:
        df_comtrade_structured = df_comtrade_structured.iloc[:, :len(column_names)]

df_comtrade_structured.columns = column_names

print(f"Structured shape: {df_comtrade_structured.shape[0]:,} rows x {df_comtrade_structured.shape[1]:,} columns")
print("Columns:", list(df_comtrade_structured.columns))
df_comtrade_structured.head()